# ECABSD — Fully Verified Clean Training (VigneshReddyKura/ecabsd)
Every cell prints explicit verification output before proceeding to the next step.


In [ ]:
# ============================================================
# CELL 1: Wipe old ghost code and clone Vignesh clean repo
# ============================================================
import os, shutil

WORK = '/kaggle/working'
REPO = 'https://github.com/VigneshReddyKura/ecabsd.git'
DEST = WORK + '/ecabsd'

os.chdir(WORK)

if os.path.exists(DEST):
    shutil.rmtree(DEST)
    print('[SETUP] Removed old ghost code.')

print('Cloning...')
!git clone {REPO} {DEST}

# Verify clone succeeded
if not os.path.exists(DEST + '/train.py'):
    raise RuntimeError('Clone failed! train.py not found.')

os.chdir(DEST)

print('PWD:', os.getcwd())
print('Repo files:', sorted(os.listdir('.')))
print('Repo cloned OK')

In [ ]:
# ============================================================
# CELL 2: Install dependencies
# ============================================================
import subprocess, sys, torch

tv = torch.__version__.split('+')[0]
cu = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'[DEPS] torch={tv}, cuda={cu}')

pkgs = ['fair-esm', 'biopython', 'pyyaml', 'scikit-learn', 'matplotlib', 'seaborn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)

pyg_url = f'https://data.pyg.org/whl/torch-{tv}+{cu}.html'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch-scatter', 'torch-sparse', 'torch-geometric', '-f', pyg_url
], check=True)

print('[DEPS] ✅ All packages installed.')

In [ ]:
# ============================================================
# CELL 3: Find and link dataset — uses data/processed + data/splits.csv
# ============================================================
import os, shutil, pandas as pd

input_root    = '/kaggle/input'
processed_src = None
splits_src    = None

print('[DATA] Scanning /kaggle/input...')
for root, dirs, files in os.walk(input_root):
    for f in files:
        if f == 'splits.csv' and splits_src is None:
            splits_src = os.path.join(root, f)
            print(f'[DATA]   Found splits.csv  : {splits_src}')
    for d in dirs:
        if d in ['processed', 'db5_processed'] and processed_src is None:
            processed_src = os.path.join(root, d)
            print(f'[DATA]   Found graphs dir  : {processed_src}')

if not processed_src:
    raise FileNotFoundError('[DATA] Cannot find processed graphs in /kaggle/input!')
if not splits_src:
    raise FileNotFoundError('[DATA] Cannot find splits.csv in /kaggle/input!')

os.makedirs('data', exist_ok=True)

# Link graphs to data/processed (NOT db5_processed)
dest_graphs = 'data/processed'
if not os.path.exists(dest_graphs):
    os.symlink(processed_src, dest_graphs)
n_graphs = len([f for f in os.listdir(dest_graphs) if f.endswith('.pt')])

# Copy splits to data/splits.csv (NOT db5_splits.csv)
dest_csv = 'data/splits.csv'
shutil.copy(splits_src, dest_csv)

df = pd.read_csv(dest_csv)
vc = df['split'].value_counts()

print()
print('=' * 50)
print('  DATA MOUNT SUMMARY')
print('=' * 50)
print(f'  Graph files     : {n_graphs} .pt files')
print(f'  Splits CSV rows : {len(df)}')
print(f'  train rows      : {vc.get("train", 0)}')
print(f'  val   rows      : {vc.get("val",   0)}')
print(f'  test  rows      : {vc.get("test",  0)}')
print('=' * 50)
print('[DATA] ✅ Dataset mounted.')

In [ ]:
# ============================================================
# CELL 4: Fix data leakage — zero overlap guaranteed
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('data/splits.csv')

# Check existing leakage
t = set(df[df['split']=='train']['pdb_id'])
v = set(df[df['split']=='val']['pdb_id'])
e = set(df[df['split']=='test']['pdb_id'])

tv = len(t & v)
te = len(t & e)
ve = len(v & e)

print('[LEAKAGE] Before fix:')
print(f'  Train-Val overlap  : {tv}')
print(f'  Train-Test overlap : {te}')
print(f'  Val-Test overlap   : {ve}')

if tv > 0 or te > 0 or ve > 0:
    print('[LEAKAGE] Leakage found — rebuilding clean splits...')

    all_pdb_ids = df['pdb_id'].unique()
    train_val, test_ids = train_test_split(all_pdb_ids, test_size=0.15, random_state=42)
    train_ids, val_ids  = train_test_split(train_val,   test_size=0.176, random_state=42)

    test_set  = set(test_ids)
    val_set   = set(val_ids)
    train_set = set(train_ids)

    def assign_split(pid):
        if pid in test_set:  return 'test'
        if pid in val_set:   return 'val'
        return 'train'

    df['split'] = df['pdb_id'].apply(assign_split)
    df.to_csv('data/splits.csv', index=False)

# Final verification
df = pd.read_csv('data/splits.csv')
t2 = set(df[df['split']=='train']['pdb_id'])
v2 = set(df[df['split']=='val']['pdb_id'])
e2 = set(df[df['split']=='test']['pdb_id'])

print()
print('[LEAKAGE] After fix:')
print(f'  Train-Val overlap  : {len(t2 & v2)}')
print(f'  Train-Test overlap : {len(t2 & e2)}')
print(f'  Val-Test overlap   : {len(v2 & e2)}')

assert len(t2 & v2) == 0, 'LEAKAGE: train/val'
assert len(t2 & e2) == 0, 'LEAKAGE: train/test'
assert len(v2 & e2) == 0, 'LEAKAGE: val/test'
print('[LEAKAGE] ✅ Zero leakage confirmed.')

In [ ]:
# ============================================================
# CELL 5: Remove bad graphs — user's exact code
# ============================================================
import torch, glob, os, shutil

SRC = 'data/processed'
BAD = 'data/bad_graphs'
os.makedirs(BAD, exist_ok=True)

good = 0
bad  = 0

for f in glob.glob(SRC + '/*.pt'):
    try:
        g = torch.load(f, map_location='cpu', weights_only=False)

        x_ok = hasattr(g, 'x') and g.x is not None and g.x.dim() == 2 and g.x.shape[1] == 33
        e_ok = hasattr(g, 'edge_attr') and g.edge_attr is not None and g.edge_attr.dim() == 2 and g.edge_attr.shape[1] == 5

        if x_ok and e_ok:
            good += 1
        else:
            bad += 1
            shutil.move(f, os.path.join(BAD, os.path.basename(f)))

    except Exception as e:
        bad += 1
        shutil.move(f, os.path.join(BAD, os.path.basename(f)))

print('Good graphs:', good)
print('Moved bad graphs:', bad)
print('Final graphs in processed:', len(glob.glob(SRC + '/*.pt')))
print('Ready for recovery step.')


In [ ]:
# ============================================================
# CELL 5.5: Auto-Recover Bad Graphs
# Downloads raw PDBs and rebuilds missing/bad graphs with x=33
# ============================================================
import sys, subprocess
print('[RECOVERY] Starting graph recovery script...')
subprocess.run([sys.executable, 'scripts/recover_graphs.py'])
print('[RECOVERY] Done.')


In [ ]:
# ============================================================
# CELL 6: Update config.yaml with final dimensions + verify
# ============================================================
import yaml, os, pandas as pd

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['processed_dir']      = 'data/processed'
cfg['data']['splits_csv']         = 'data/splits.csv'
cfg['model']['esm_dim']           = 33      # match actual node feature dim
cfg['model']['edge_feature_dim']  = 5   # match actual edge feature dim
cfg['training']['epochs']         = 100
cfg['training']['num_workers']    = 0

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Print verification
final_cfg = yaml.safe_load(open('config.yaml'))
print('Config [data] section after update:')
print(f"  processed_dir  : {final_cfg['data']['processed_dir']}")
print(f"  splits_csv     : {final_cfg['data']['splits_csv']}")
print()
print('Config [model] section after update:')
print(f"  esm_dim        : {final_cfg['model'].get('esm_dim')}")
print(f"  edge_feature_dim: {final_cfg['model'].get('edge_feature_dim')}")
print(f"  hidden_dim     : {final_cfg['model'].get('hidden_dim')}")
print()

df = pd.read_csv(final_cfg['data']['splits_csv'])
vc = df['split'].value_counts()
print('Usable sample counts:')
print(f"  train : {vc.get('train', 0)}")
print(f"  val   : {vc.get('val',   0)}")
print(f"  test  : {vc.get('test',  0)}")
print()

t = set(df[df['split']=='train']['pdb_id'])
v = set(df[df['split']=='val']['pdb_id'])
e = set(df[df['split']=='test']['pdb_id'])
print('Leakage check:')
print(f'  Train-Val overlap  : {len(t & v)}')
print(f'  Train-Test overlap : {len(t & e)}')
print(f'  Val-Test overlap   : {len(v & e)}')
print()

assert vc.get('train', 0) >= 1000, f'Too few train samples: {vc.get("train",0)}'
assert vc.get('val',   0) >= 100,  f'Too few val samples: {vc.get("val",0)}'
assert len(t & v) == 0, 'LEAKAGE: train/val'
assert len(t & e) == 0, 'LEAKAGE: train/test'
assert len(v & e) == 0, 'LEAKAGE: val/test'

print('All checks passed — safe to train!')

In [ ]:
# ============================================================
# CELL 7: START TRAINING — watch for 'Loaded ~1800 train'
# ============================================================
import subprocess, sys

print('[TRAIN] Starting 100-epoch training on VigneshReddyKura/ecabsd...')
print('[TRAIN] Expect: [Dataset] Loaded ~1800 train  |  ~360 val')
print()

result = subprocess.run(
    [sys.executable, 'train.py'],
    cwd='/kaggle/working/ecabsd'
)

if result.returncode != 0:
    print('[TRAIN] ❌ Training exited with errors. Check the output above.')
else:
    print('[TRAIN] ✅ Training completed successfully!')

In [ ]:
# ============================================================
# CELL 8: Evaluate on test set
# ============================================================
import subprocess, sys

print('[EVAL] Running evaluation on clean test set...')
subprocess.run([sys.executable, 'evaluate.py'], cwd='/kaggle/working/ecabsd')
print('[EVAL] ✅ Done.')

In [ ]:
# ============================================================
# CELL 9: Verify timestamp + package everything for download
# ============================================================
import os, time, zipfile

ckpt = 'checkpoints/best_model.pt'
hist = 'logs/training_history.json'

print('Checkpoint verification:')
if os.path.exists(ckpt):
    print(f'  best_model.pt    : {time.ctime(os.path.getmtime(ckpt))}  ({os.path.getsize(ckpt)//1024} KB)')
else:
    print('  ❌ best_model.pt NOT FOUND — training may have failed!')

if os.path.exists(hist):
    import json
    history = json.load(open(hist))
    best_ep = max(history, key=lambda x: x['val']['f1'])
    print(f'  training_history : {time.ctime(os.path.getmtime(hist))}')
    print(f'  Best val F1      : {best_ep["val"]["f1"]:.4f} @ epoch {best_ep["epoch"]}')

# Package everything
zip_path = '/kaggle/working/ecabsd_vignesh_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['checkpoints', 'results', 'logs', 'models']:
        if os.path.isdir(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    zf.write(os.path.join(root, file))

print(f'\n✅ Download ready: {zip_path}')
print('Extract and copy models/ + checkpoints/ into your local ecabsd folder.')